# CUTEst

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.experiment.for_cutest_run import (
    CUTEstTask,
    get_file_path,
    load_npz,
    load_npz_with_metadata,
    run_tasks,
)
from qnlab.experiment.for_cutest_vis import draw_data_profile, draw_pp
from qnlab.util.method import COLORS, LINE_STYLES, Method, get_methods

## Experiment workflow

This single workflow covers the floating-point experiments and all explicit-noise experiments reported in the paper. Results are stored under `data/temp/<scenario>/seed_<seed>/` and overwrite earlier results for the same condition. A full run is long-running.

In [ ]:
working_directory = Path.cwd().resolve()
if (working_directory / "pyproject.toml").exists():
    repository_root = working_directory
elif (working_directory.parent / "pyproject.toml").exists():
    repository_root = working_directory.parent
else:
    raise RuntimeError("Open this notebook from the repository root or notebooks/.")
os.chdir(repository_root)
print(repository_root)

In [ ]:
SCENARIOS = {
    "float64": {
        "precision": 64,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": None,
        "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float32": {
        "precision": 32,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": None,
        "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    "float16": {
        "precision": 16,
        "function_noise": 0.0,
        "gradient_noise": 0.0,
        "assumed_function_error": None,
        "solver_gtol": None,
        "score_gtols": [1e-1, 1e-3, 1e-5],
    },
    # Theory-aligned experiment: the gradient oracle is exact.
    "function_only": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    # Deliberately adverse stress test outside the exact-gradient theorem.
    "joint_noise": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 1e-3,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_under": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-4,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_nominal": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-2,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
    "eps_over": {
        "precision": 64,
        "function_noise": 1e-3,
        "gradient_noise": 0.0,
        "assumed_function_error": 1e-1,
        "solver_gtol": 1e-2,
        "score_gtols": [1e-2],
    },
}

# SAFETY: the checked-in default is an intentionally tiny pilot, never the full suite.
PILOT_PROBLEMS = ["AKIVA", "ARWHEAD", "ROSENBR"]
PILOT_METHODS = ["NTRQN", "OFFO"]
FULL_BENCHMARK = os.getenv("QNLAB_FULL_CUTEST", "0") == "1"
DEFAULT_RESPONSE_SCENARIOS = [
    "function_only",
    "joint_noise",
    "eps_under",
    "eps_nominal",
    "eps_over",
]
DEFAULT_SCENARIOS = DEFAULT_RESPONSE_SCENARIOS if FULL_BENCHMARK else ["float64"]
SCENARIOS_TO_RUN = os.getenv("QNLAB_SCENARIOS", ",".join(DEFAULT_SCENARIOS)).split(",")
NOISY_SEEDS = [int(seed) for seed in os.getenv("QNLAB_SEEDS", "0,1,2,3,4").split(",")]
PROBLEMS_TO_RUN = None if FULL_BENCHMARK else PILOT_PROBLEMS
METHODS_TO_RUN = None if FULL_BENCHMARK else PILOT_METHODS
TIME_LIMIT = 600.0
MAX_ITERATIONS = 15_000
RUN_EXPERIMENTS = os.getenv("QNLAB_RUN_CUTEST", "0") == "1"
OVERWRITE_EXISTING = os.getenv("QNLAB_OVERWRITE_CUTEST", "0") == "1"
ERROR_CAUSING_TASKS = [
    (16, "INDEFM", "SciPy"),
    (16, "INDEFM", "NTRQN"),
    (16, "INDEFM", "NTRQN-MS"),
    (16, "INDEFM", "Reg-Sec"),
    (16, "OSCIGRAD", "NTRQN-MS"),
    (32, "INDEFM", "NTRQN"),
    (32, "INDEFM", "NTRQN-MS"),
    (32, "OSCIGRAD", "NTRQN-MS"),
]

## Methods and execution

The checked-in default is deliberately restricted to a three-problem pilot: 64-bit arithmetic without artificial noise, seed `0`, and only `NTRQN` versus the standalone first-order `OFFO` baseline. It never selects all CUTEst problems. The notebook only inspects this six-task list unless `QNLAB_RUN_CUTEST=1` is set. A full benchmark additionally requires the conspicuous opt-in `QNLAB_FULL_CUTEST=1`; use `QNLAB_SCENARIOS` and `QNLAB_SEEDS` to split it, and `QNLAB_OVERWRITE_CUTEST=1` only to replace existing results. Noisy scenarios use five paired seeds (`0` through `4`), while deterministic precision scenarios use only seed `0`. The two NTQN stopping rules are retained as separate methods until their empirical behavior has been compared. The restart ablation permits at most ten restarts, and the stored diagnostics record the actual restart count for each run.

In [ ]:
STANDARD_LABELS = {
    "NTRQN",
    "NTRQN-MS",
    "Line",
    "Line-MS",
    "OFFO",
    "Reg",
    "Reg-Sec",
    "SciPy",
    "NTQN",
}
SCENARIO_DEFAULT_LABELS = {
    "function_only": STANDARD_LABELS
    | {"NTRQN-OFFO", "NTRQN-Restart", "NTQN-Default-Termination"},
    "joint_noise": STANDARD_LABELS | {"NTQN-Default-Termination"},
    "eps_under": {"NTRQN", "NTRQN-MS"},
    "eps_nominal": {"NTRQN", "NTRQN-MS"},
    "eps_over": {"NTRQN", "NTRQN-MS"},
}
for scenario in ("float64", "float32", "float16"):
    SCENARIO_DEFAULT_LABELS[scenario] = STANDARD_LABELS

In [ ]:
def get_experiment_methods(max_iterations, solver_gtol=None):
    methods, _, _ = get_methods(m=10, MI=max_iterations)
    methods.extend(
        [
            (
                Method("OFFO", label="OFFO"),
                {"max_iterations": max_iterations},
            ),
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-OFFO"),
                {"m": 10, "max_iterations": max_iterations, "force_offo": 1},
            ),
            (
                Method("NTRQN", "cautious", "damped", "bfgs", label="NTRQN-Restart"),
                {
                    "m": 10,
                    "max_iterations": max_iterations,
                    "restart_threshold": 1.0,
                    "max_restarts": 10,
                },
            ),
            (
                Method("NTQN", "raw", "raw", "bfgs", label="NTQN-Default-Termination"),
                {
                    "m": 10,
                    "max_iterations": max_iterations,
                    "terminate": 3,
                    "stop_at_gtol": 0,
                },
            ),
        ]
    )
    if solver_gtol is not None:
        methods = [
            (method, option | {"gtol": solver_gtol}) for method, option in methods
        ]
    return methods


def get_problem_names(precision, selected=None):
    names = problemsToRun(precision)
    if selected is None:
        return names
    unknown = sorted(set(selected) - set(names))
    if unknown:
        raise ValueError(f"Problems not in the {precision}-bit valid list: {unknown}")
    return selected


def get_scenario_seeds(scenario):
    config = SCENARIOS[scenario]
    has_noise = config["function_noise"] > 0.0 or config["gradient_noise"] > 0.0
    return NOISY_SEEDS if has_noise else [0]


def make_task(scenario, seed, problem_name, method, options):
    config = SCENARIOS[scenario]
    assumed_error = config["assumed_function_error"]
    return CUTEstTask(
        problem_name=problem_name,
        method=method,
        options=options,
        precision=config["precision"],
        function_noise=np.float64(config["function_noise"]),
        gradient_noise=np.float64(config["gradient_noise"]),
        assumed_function_error=(
            None if assumed_error is None else np.float64(assumed_error)
        ),
        seed=seed,
        scenario=scenario,
    )


def load_scenario_results(scenario, seeds, gtol, labels=None):
    config = SCENARIOS[scenario]
    problems = get_problem_names(config["precision"], PROBLEMS_TO_RUN)
    method_options = get_experiment_methods(MAX_ITERATIONS, config["solver_gtol"])
    labels = labels or METHODS_TO_RUN or SCENARIO_DEFAULT_LABELS[scenario]
    method_options = [entry for entry in method_options if entry[0].label in labels]
    alg_names = [method.label for method, _ in method_options]
    instances = [(seed, problem) for seed in seeds for problem in problems]
    calls = np.full((len(method_options), len(instances)), np.inf)
    dimensions = np.full(len(instances), np.nan)
    metadata = np.empty((len(method_options), len(instances)), dtype=object)
    metadata.fill(None)
    for instance_index, (seed, problem) in enumerate(instances):
        for method_index, (method, options) in enumerate(method_options):
            task = make_task(scenario, seed, problem, method, options)
            callback, run_metadata = load_npz_with_metadata(task, verbose=False)
            metadata[method_index, instance_index] = run_metadata
            if "dimension" in run_metadata:
                dimensions[instance_index] = run_metadata["dimension"]
            reached = np.flatnonzero(np.asarray(callback.gnorms) <= gtol)
            if reached.size > 0:
                calls[method_index, instance_index] = max(1, callback.calls[reached[0]])
    instance_names = [f"{problem} (seed={seed})" for seed, problem in instances]
    return alg_names, calls, instance_names, dimensions, metadata


def summarize_run_metadata(alg_names, metadata):
    rows = []
    for method_index, method in enumerate(alg_names):
        entries = [entry for entry in metadata[method_index] if entry]
        statuses = [entry.get("status", "unrecorded") for entry in entries]
        return_values = [entry.get("return_code_value") for entry in entries]
        rows.append(
            {
                "method": method,
                "result files": len(entries),
                "missing files": metadata.shape[1] - len(entries),
                "completed": statuses.count("completed"),
                "timeouts": statuses.count("timeout"),
                "exceptions": statuses.count("error"),
                "error return codes": sum(
                    value is not None and value < 0 for value in return_values
                ),
            }
        )
    return pd.DataFrame(rows).set_index("method")


def summarize_diagnostics(alg_names, metadata):
    restart_rows = []
    ntqn_rows = []
    for method_index, method in enumerate(alg_names):
        diagnostics = [
            entry.get("diagnostics", {})
            for entry in metadata[method_index]
            if entry
        ]
        if method == "NTRQN-Restart" and diagnostics:
            counts = np.asarray(
                [entry.get("OFFO accumulator restart", 0) for entry in diagnostics]
            )
            restart_rows.append(
                {
                    "method": method,
                    "mean": counts.mean(),
                    "median": np.median(counts),
                    "maximum": counts.max(),
                    "cap reached": np.count_nonzero(counts >= 10),
                }
            )
        for entry in diagnostics:
            if "NTQN termination flag" in entry:
                ntqn_rows.append(
                    {"method": method, "flag": entry["NTQN termination flag"]}
                )
    restart_summary = pd.DataFrame(restart_rows)
    ntqn_summary = (
        pd.crosstab(
            pd.Series([row["method"] for row in ntqn_rows], name="method"),
            pd.Series([row["flag"] for row in ntqn_rows], name="flag"),
        )
        if ntqn_rows
        else pd.DataFrame()
    )
    return restart_summary, ntqn_summary

In [ ]:
unknown_scenarios = sorted(set(SCENARIOS_TO_RUN) - set(SCENARIOS))
if unknown_scenarios:
    raise ValueError(f"Unknown scenarios: {unknown_scenarios}")

tasks = []
for scenario in SCENARIOS_TO_RUN:
    config = SCENARIOS[scenario]
    selected_problems = get_problem_names(config["precision"], PROBLEMS_TO_RUN)
    selected_method_options = get_experiment_methods(
        MAX_ITERATIONS, config["solver_gtol"]
    )
    if METHODS_TO_RUN is not None:
        selected_method_options = [
            entry
            for entry in selected_method_options
            if entry[0].label in METHODS_TO_RUN
        ]
        missing = sorted(
            set(METHODS_TO_RUN) - {entry[0].label for entry in selected_method_options}
        )
        if missing:
            raise ValueError(f"Unknown method labels: {missing}")
    else:
        selected_method_options = [
            entry
            for entry in selected_method_options
            if entry[0].label in SCENARIO_DEFAULT_LABELS[scenario]
        ]
    tasks.extend(
        make_task(scenario, seed, problem, method, options)
        for seed in get_scenario_seeds(scenario)
        for problem in selected_problems
        for method, options in selected_method_options
    )
mode = "FULL BENCHMARK" if FULL_BENCHMARK else "TINY PILOT (3 problems only)"
print(f"MODE: {mode}")
print(f"Scenarios: {SCENARIOS_TO_RUN}")
print(
    f"Problems: {PROBLEMS_TO_RUN if PROBLEMS_TO_RUN is not None else 'ALL VALID PROBLEMS'}"
)
print(
    f"Methods: {METHODS_TO_RUN if METHODS_TO_RUN is not None else 'SCENARIO DEFAULTS'}"
)
print(f"Prepared {len(tasks)} tasks.")
for task in tasks[:20]:
    path = get_file_path(task)
    print(
        f"{task.scenario:13s} seed={task.seed} {task.problem_name:24s} "
        f"{task.method.label:18s} -> {path}"
    )
if len(tasks) > 20:
    print(f"... {len(tasks) - 20} additional tasks omitted from this preview.")

if RUN_EXPERIMENTS:
    run_tasks(tasks, ERROR_CAUSING_TASKS, int(TIME_LIMIT), overwrite=OVERWRITE_EXISTING)
else:
    print("Dry run only. Set QNLAB_RUN_CUTEST=1 to execute these tasks.")

## Visualize saved results

Select scenarios after their runs have completed. The default empty list avoids overwriting existing figures accidentally.

In [ ]:
SCENARIOS_TO_PLOT = [
    scenario
    for scenario in os.getenv("QNLAB_PLOT_SCENARIOS", "").split(",")
    if scenario
]

def display_result_summaries(alg_names, calls, metadata):
    best_calls = calls.min(axis=0)
    fastest = (calls == best_calls) & np.isfinite(best_calls)[None, :]
    display(
        pd.DataFrame(
            {
                "solved (percent)": 100.0 * np.isfinite(calls).mean(axis=1),
                "fastest, ties included (percent)": 100.0 * fastest.mean(axis=1),
            },
            index=alg_names,
        )
    )
    display(summarize_run_metadata(alg_names, metadata))
    restart_summary, ntqn_summary = summarize_diagnostics(alg_names, metadata)
    if not restart_summary.empty:
        display(restart_summary.set_index("method"))
    if not ntqn_summary.empty:
        display(ntqn_summary)


_, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
ALGORITHM_COLORS["OFFO"] = COLORS["OFFO"]
ALGORITHM_LINE_STYLES["OFFO"] = LINE_STYLES["OFFO"]
for scenario in SCENARIOS_TO_PLOT:
    config = SCENARIOS[scenario]
    for gtol in config["score_gtols"]:
        seeds = get_scenario_seeds(scenario)
        primary_labels = METHODS_TO_RUN or (
            SCENARIO_DEFAULT_LABELS[scenario] - {"NTRQN-Restart"}
        )
        alg_names, calls, instances, dimensions, metadata = load_scenario_results(
            scenario, seeds, gtol, labels=primary_labels
        )
        output_path = None
        if scenario not in {"float64", "float32", "float16", "joint_noise"}:
            gtol_name = f"{gtol:.0e}".replace("+", "")
            output_path = (
                Path("doc/imgs/compare") / f"_pp_{scenario}_gtol{gtol_name}.pdf"
            )
        draw_pp(
            alg_names,
            calls,
            ALGORITHM_COLORS,
            ALGORITHM_LINE_STYLES,
            config["precision"],
            np.float64(max(config["function_noise"], config["gradient_noise"])),
            np.float64(gtol),
            output_path=output_path,
        )
        gtol_name = f"{gtol:.0e}".replace("+", "")
        data_profile_path = (
            Path("doc/imgs/compare") / f"_dp_{scenario}_gtol{gtol_name}.pdf"
        )
        if np.all(np.isfinite(dimensions)):
            draw_data_profile(
                alg_names,
                calls,
                dimensions,
                ALGORITHM_COLORS,
                ALGORITHM_LINE_STYLES,
                data_profile_path,
            )
        else:
            print("Skipping data profile because some result metadata lack dimensions.")

        table = pd.DataFrame(calls.T, index=instances, columns=alg_names)
        display(table)

        display_result_summaries(alg_names, calls, metadata)

        if len(seeds) > 1:
            solved_by_seed = []
            for seed in seeds:
                _, seed_calls, _, _, _ = load_scenario_results(
                    scenario, [seed], gtol, labels=primary_labels
                )
                solved_by_seed.append(np.isfinite(seed_calls).mean(axis=1))
            solved_by_seed = np.asarray(solved_by_seed)
            seed_summary = pd.DataFrame(
                {
                    "mean solved fraction": solved_by_seed.mean(axis=0),
                    "std. dev. across seeds": solved_by_seed.std(axis=0, ddof=1),
                },
                index=alg_names,
            )
            display(seed_summary)

        if scenario == "function_only" and METHODS_TO_RUN is None:
            restart_labels = {"NTRQN", "NTRQN-Restart"}
            restart_names, restart_calls, _, restart_dimensions, restart_metadata = (
                load_scenario_results(
                    scenario, seeds, gtol, labels=restart_labels
                )
            )
            restart_stem = f"function_only_restart_gtol{gtol_name}"
            draw_pp(
                restart_names,
                restart_calls,
                ALGORITHM_COLORS,
                ALGORITHM_LINE_STYLES,
                config["precision"],
                np.float64(config["function_noise"]),
                np.float64(gtol),
                output_path=Path("doc/imgs/compare") / f"_pp_{restart_stem}.pdf",
            )
            if np.all(np.isfinite(restart_dimensions)):
                draw_data_profile(
                    restart_names,
                    restart_calls,
                    restart_dimensions,
                    ALGORITHM_COLORS,
                    ALGORITHM_LINE_STYLES,
                    Path("doc/imgs/compare") / f"_dp_{restart_stem}.pdf",
                )
            display_result_summaries(
                restart_names, restart_calls, restart_metadata
            )